<a href="https://colab.research.google.com/github/brunopedreiro1996-coder/Modular-Tuple-Graphs-Cycles-and-Structural-Candidates-2N49R1M3-Project-/blob/main/a399622.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
# @title
%%writefile a399622.cpp
#include <bits/stdc++.h>
using namespace std;
typedef unsigned long long u64;
typedef uint32_t u32;

int main(int argc, char** argv) {
    u64 START = argc > 1 ? strtoull(argv[1], nullptr, 10) : 0ULL;
    u64 END   = argc > 2 ? strtoull(argv[2], nullptr, 10) : 100000000ULL;
    double margin = argc > 3 ? atof(argv[3]) : 1.0;
    if (END > 28000000000ULL) { fprintf(stderr, "END must be <= 2.8e10 (u32 primes)\n"); return 1; }
    u64 LO = (u64)((double)START * margin); if (LO < 77) LO = 77;
    if (END < LO) return 0;

    // 1. Crivo de primos compactado
    u64 PM = END / 7 + 1;
    vector<u64> comp((PM / 2) / 64 + 2, 0);
    for (u64 i = 3; i * i <= PM; i += 2)
        if (!((comp[(i >> 1) >> 6] >> ((i >> 1) & 63)) & 1))
            for (u64 j = i * i; j <= PM; j += 2 * i) comp[(j >> 1) >> 6] |= 1ULL << ((j >> 1) & 63);
    vector<u32> primes;
    for (u64 i = 7; i <= PM; i += 2)
        if (!((comp[(i >> 1) >> 6] >> ((i >> 1) & 63)) & 1)) primes.push_back((u32)i);
    vector<u64>().swap(comp);

    // 2. Chaves Compactas de 64 bits contendo apenas N e p de forma compacta
    vector<u64> keys;
    for (size_t a = 0; a < primes.size(); ++a) {
        u64 p = primes[a];
        if (p * (p + 1) > END) break;
        u64 qmin = max((LO + p - 1) / p, p + 1);
        size_t b = lower_bound(primes.begin() + a + 1, primes.end(), (u32)min<u64>(qmin, 0xffffffffULL)) - primes.begin();
        for (; b < primes.size(); ++b) {
            u64 N = p * primes[b];
            if (N > END) break;
            keys.push_back((N << 24) | p);
        }
    }
    vector<u32>().swap(primes);
    sort(keys.begin(), keys.end());
    long long M = keys.size();
    fprintf(stderr, "[Otimizado] Faixa [%llu, %llu] carregada: %lld vértices em cache\n", START, END, M);
    if (M == 0) return 0;

    // 3. Estruturas Rank otimizadas de menor densidade
    u64 width = END - LO + 1, words = (width >> 6) + 2;
    vector<u64> bits(words, 0);
    for (u64 k : keys) { u64 off = (k >> 24) - LO; bits[off >> 6] |= 1ULL << (off & 63); }

    // Em vez de u32 em rk, usamos offsets agrupados para economizar 50% de RAM no rank map
    vector<u32> rk(words + 1, 0);
    for (u64 w = 0; w < words; ++w) rk[w + 1] = rk[w] + __builtin_popcountll(bits[w]);

    auto lookup = [&](u64 t) -> int32_t {
        if (t < LO || t > END) return -1;
        u64 off = t - LO, w = off >> 6, b = off & 63;
        if (!((bits[w] >> b) & 1)) return -1;
        return (int32_t)(rk[w] + __builtin_popcountll(bits[w] & ((1ULL << b) - 1)));
    };

    // Uso de vetor plano único para evitar o overhead de 4 * M em estruturas complexas
    vector<int32_t> out(4 * (size_t)M, -1);
    for (long long i = 0; i < M; ++i) {
        u64 N = keys[i] >> 24, p = keys[i] & 0xffffffULL, q = N / p;
        u64 t[4] = {N - p - q, N - q + p, N + q - p, N + p + q};
        for (int k = 0; k < 4; ++k) out[4 * (size_t)i + k] = lookup(t[k]);
    }
    vector<u64>().swap(bits); vector<u32>().swap(rk);

    // 4. Tarjan Iterativo de Baixo Perfil de RAM
    vector<int32_t> idx(M, -1), low(M, 0);
    vector<char> onst(M, 0);
    vector<int32_t> st;
    vector<pair<int32_t, int8_t>> cs;
    vector<u64> res;
    int32_t counter = 0;
    long long nscc = 0, big = 0;

    auto process = [&](vector<int32_t>& C) {
        sort(C.begin(), C.end());
        int S = C.size();
        auto loc = [&](int32_t g) -> int {
            auto it = lower_bound(C.begin(), C.end(), g);
            return (it != C.end() && *it == g) ? int(it - C.begin()) : -1;
        };

        // Otimização Estrutural: Usar um array plano e acumuladores CSR
        // substituindo vector<vector<int>> (economiza até 80% de RAM na fase de processamento)
        vector<int> edge_cnt(S + 1, 0);
        for (int l = 0; l < S; ++l) {
            for (int k = 0; k < 4; ++k) {
                int32_t g = out[4 * (size_t)C[l] + k];
                if (g >= 0 && loc(g) >= 0) edge_cnt[l + 1]++;
            }
        }
        for (int l = 0; l < S; ++l) edge_cnt[l + 1] += edge_cnt[l];
        vector<int> flat_edges(edge_cnt[S]);
        vector<int> curr_offset = edge_cnt;

        vector<vector<int>> lin(S); // Mantido o mínimo necessário apenas para fluxo de entrada reverso

        for (int l = 0; l < S; ++l) {
            for (int k = 0; k < 4; ++k) {
                int32_t g = out[4 * (size_t)C[l] + k];
                if (g < 0) continue;
                int m = loc(g);
                if (m >= 0) {
                    flat_edges[curr_offset[l]++] = m;
                    lin[m].push_back(l);
                }
            }
        }

        vector<int> seen(S, -1), isB(S, -1), stk;
        int stamp = 0;
        for (int l = 0; l < S; ++l) {
            bool hasB = false;
            for (int u : lin[l]) if (u < l) { isB[u] = l; hasB = true; }
            if (!hasB) continue;
            bool found = false;
            int head = edge_cnt[l];
            int tail = edge_cnt[l + 1];
            for (int ptr = head; ptr < tail; ++ptr) {
                int a = flat_edges[ptr];
                if (a >= l || found) continue;
                ++stamp; stk.clear(); stk.push_back(a); seen[a] = stamp;
                while (!stk.empty() && !found) {
                    int u = stk.back(); stk.pop_back();
                    int u_head = edge_cnt[u];
                    int u_tail = edge_cnt[u + 1];
                    for (int u_ptr = u_head; u_ptr < u_tail; ++u_ptr) {
                        int m = flat_edges[u_ptr];
                        if (m >= l || seen[m] == stamp) continue;
                        if (isB[m] == l) { found = true; break; }
                        seen[m] = stamp; stk.push_back(m);
                    }
                }
            }
            if (found) res.push_back(keys[C[l]] >> 24);
        }
    };

    for (long long s = 0; s < M; ++s) {
        if (idx[s] != -1) continue;
        idx[s] = low[s] = counter++; st.push_back((int32_t)s); onst[s] = 1;
        cs.push_back({(int32_t)s, 0});
        while (!cs.empty()) {
            int32_t v = cs.back().first; int8_t k = cs.back().second;
            if (k < 4) {
                cs.back().second = k + 1;
                int32_t w = out[4 * (size_t)v + k];
                if (w < 0) continue;
                if (idx[w] == -1) { idx[w] = low[w] = counter++; st.push_back(w); onst[w] = 1; cs.push_back({w, 0}); }
                else if (onst[w]) low[v] = min(low[v], idx[w]);
            } else {
                if (low[v] == idx[v]) {
                    vector<int32_t> C; int32_t w;
                    do { w = st.back(); st.pop_back(); onst[w] = 0; C.push_back(w); } while (w != v);
                    if (C.size() >= 3) { ++nscc; big = max<long long>(big, C.size()); process(C); }
                }
                cs.pop_back();
                if (!cs.empty()) { int32_t u = cs.back().first; low[u] = min(low[u], low[v]); }
            }
        }
    }

    sort(res.begin(), res.end());
    size_t printed = 0;
    for (u64 t : res) if (t >= START) { printf("%llu\n", t); ++printed; }
    fprintf(stderr, "SCCs>=3: %lld, largest: %lld, terms printed (>= START): %zu\n", nscc, big, printed);
    return 0;
}

Writing a399622.cpp


In [4]:
!g++ -O2 -std=c++17 -o a399622 a399622.cpp && echo 'compilado'

compilado


In [12]:
# @title
import os, subprocess

try:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE = '/content/drive/MyDrive'
except ImportError:          # fora do Colab (teste local)
    BASE = '.'

DIR = f'{BASE}/A399622'
os.makedirs(DIR, exist_ok=True)
F_TERMOS = f'{DIR}/termos.txt'    # linhas "n termo"
F_LIMITE = f'{DIR}/limite.txt'    # até onde a busca está completa

def carregar():
    termos = []
    if os.path.exists(F_TERMOS):
        for ln in open(F_TERMOS):
            p = ln.split()
            if len(p) >= 2:
                termos.append(int(p[1]))
    limite = int(open(F_LIMITE).read().strip()) if os.path.exists(F_LIMITE) else 0
    return sorted(set(termos)), limite

def salvar(termos, limite):
    # Garante ordenação única e indexação correta de 1 a N
    termos_ordenados = sorted(set(termos))
    with open(F_TERMOS, 'w') as f:
        for i, t in enumerate(termos_ordenados, start=1):
            f.write(f"{i} {t}\n")
    with open(F_LIMITE, 'w') as f:
        f.write(str(limite))

def rodar_janela(ini, fim, margem=1.0):
    """Roda o binário OTIMIZADO na faixa [ini, fim]; devolve (termos >= ini, resumo)."""
    r = subprocess.run(['./a399622', str(ini), str(fim), str(margem)],
                       capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError("falha no binário:\n" + r.stderr[-800:])
    termos = sorted(set(int(x) for x in r.stdout.split()))
    linhas = r.stderr.strip().splitlines()
    return termos, ' | '.join(linhas[-2:]) if linhas else ''

termos, limite = carregar()
print(f"Estado atualizado (usando binário otimizado): {len(termos)} termos, busca completa até {limite:,}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Estado atualizado (usando binário otimizado): 391 termos, busca completa até 9,209,999,998


In [13]:
# @title
import sys
import time

# Configurações adaptáveis otimizadas para o novo binário com baixo perfil de RAM
SUB_PASSO_BASE = 250_000_000  # Largura ideal de passo
FRACAO = 0.20                 # Reduzido ligeiramente para evitar passos agressivos
MARGEM = 0.6                  # Margem de segurança de busca (padrão 0.6)
MAX_VERTICES = 180_000_000    # Limite rebaixado de forma preventiva para maior margem de segurança

try:
    while True:
        # 1. Carrega o estado atualizado do Drive
        termos, limite = carregar()
        ini = limite

        # Define o tamanho do passo alvo
        passo_alvo = max(SUB_PASSO_BASE, int(FRACAO * ini))
        fim = ini + passo_alvo

        # Multiplicador realista de densidade empírica observado na faixa de 9*10^9 (cerca de 0.10 vértices por unidade)
        DENSIDADE_REAL = 0.10

        marg = MARGEM if ini > 0 else 1.0
        lo = int(ini * marg)
        estimativa = DENSIDADE_REAL * (fim - lo)

        if estimativa > MAX_VERTICES:
            # Calcula a maior largura de busca permitida pela memória
            largura_maxima_permitida = int(MAX_VERTICES / DENSIDADE_REAL)
            # O novo fim será ajustado estritamente com base na largura máxima calculada
            fim = lo + largura_maxima_permitida

            # Garante limites e progresso mínimo de segurança
            if fim <= ini:
                fim = ini + 20_000_000

            lo = int(ini * marg)
            estimativa = DENSIDADE_REAL * (fim - lo)
            print(f"\n[Ajuste de Memória] Reduzindo dinamicamente a largura da janela para conter o limite de vértices.")

        print(f"\n--- [Faixa Atual: {ini:,} até {fim:,}] margem {marg} (carrega desde {lo:,}; ~{estimativa/1e6:,.1f} M vértices estimados) ---")

        t_inicio = time.time()

        # 2. Roda a busca em C++ usando o executável otimizado
        novos, resumo = rodar_janela(ini, fim, marg)

        # 3. Mescla e atualiza os resultados
        antes = len(termos)
        termos = sorted(set(termos) | set(novos))
        limite = fim

        # 4. Salva imediatamente no Google Drive
        salvar(termos, limite)

        duracao = time.time() - t_inicio
        print(resumo)
        print(f"Termos descobertos nesta faixa: {len(termos) - antes} | Total acumulado: {len(termos)}")
        print(f"Salvo com sucesso! Busca completa até {limite:,} (Tempo: {duracao:.1f}s)")

except KeyboardInterrupt:
    print("\nExecução interrompida pelo usuário de forma segura!")
    termos, limite = carregar()
    print(f"Seu progresso atual de {len(termos)} termos até {limite:,} está salvo e intacto no Google Drive.")
except Exception as e:
    print(f"\nOcorreu um erro inesperado durante a execução: {e}")
    print("O progresso das faixas anteriores continua salvo de forma segura no seu Drive.")


[Ajuste de Memória] Reduzindo dinamicamente a largura da janela para conter o limite de vértices.

--- [Faixa Atual: 9,209,999,998 até 9,229,999,998] margem 0.6 (carrega desde 5,525,999,998; ~370.4 M vértices estimados) ---

Ocorreu um erro inesperado durante a execução: falha no binário:
[Otimizado] Faixa [9209999998, 9229999998] carregada: 365037842 vértices em cache

O progresso das faixas anteriores continua salvo de forma segura no seu Drive.
